<a href="https://colab.research.google.com/github/Subuktageen-Farooqi/ms_course_deeplearning/blob/main/ms_deeplearning_tutorial_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning Tensorflow Implementation

In [ ]:
# Step 1 - Import Libraries
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.vgg16 import decode_predictions as decode_vgg
from tensorflow.keras.applications.resnet50 import decode_predictions as decode_resnet
import numpy as np


# Step 2 - Load and preprocess image

def load_and_preprocess_image(img_path, model_name):
    img = image.load_img(img_path, target_size=(224, 224))  # Resize
    img_array = image.img_to_array(img)  # To array
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dim

    if model_name == 'vgg':
        img_array = preprocess_vgg(img_array)
    elif model_name == 'resnet':
        img_array = preprocess_resnet(img_array)
    else:
        raise ValueError("model_name must be 'vgg' or 'resnet'")

    return img_array


# Predict top 5 classes

def predict_top_5(model, img_array, model_name):
    preds = model.predict(img_array)

    if model_name == 'vgg':
        return decode_vgg(preds, top=5)
    elif model_name == 'resnet':
        return decode_resnet(preds, top=5)
    else:
        raise ValueError("model_name must be 'vgg' or 'resnet'")


# Print predictions

def print_predictions(predictions, model_name):
    print(f"\nTop 5 predictions from {model_name.upper()}:")
    for i, pred in enumerate(predictions[0]):
        print(f"{i+1}. {pred[1]} ({pred[2]*100:.2f}%)")


def main():
    img_path = input("Enter image path: ").strip()

    # Step 3 - Load Pretrained Models
    vgg_model = VGG16(weights='imagenet')
    resnet_model = ResNet50(weights='imagenet')

    # Step 4 - Make Predictions
    # VGG16 predictions
    vgg_img = load_and_preprocess_image(img_path, 'vgg')
    vgg_preds = predict_top_5(vgg_model, vgg_img, 'vgg')
    print_predictions(vgg_preds, 'vgg16')

    # ResNet50 predictions
    resnet_img = load_and_preprocess_image(img_path, 'resnet')
    resnet_preds = predict_top_5(resnet_model, resnet_img, 'resnet')
    print_predictions(resnet_preds, 'resnet50')


if __name__ == "__main__":
    main()

# Task 1: Transfer Learning PyTorch Implementation

In [ ]:
import torch
from torchvision import models
from torchvision.models import VGG16_Weights, ResNet50_Weights
from PIL import Image


def load_image(image_path: str, transform):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0)
    return tensor


def print_top5(model, inputs, labels, model_name: str):
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.nn.functional.softmax(logits[0], dim=0)
        top_probs, top_idxs = torch.topk(probs, k=5)

    print(f"\nTop 5 predictions for {model_name}:")
    for rank, (idx, prob) in enumerate(zip(top_idxs.tolist(), top_probs.tolist()), start=1):
        print(f"{rank}. {labels[idx]} ({prob * 100:.2f}% probability)")


def main():
    image_path = input("Enter image path: ").strip()

    vgg_weights = VGG16_Weights.DEFAULT
    resnet_weights = ResNet50_Weights.DEFAULT

    vgg16 = models.vgg16(weights=vgg_weights)
    resnet50 = models.resnet50(weights=resnet_weights)

    vgg_inputs = load_image(image_path, vgg_weights.transforms())
    resnet_inputs = load_image(image_path, resnet_weights.transforms())

    print_top5(vgg16, vgg_inputs, vgg_weights.meta["categories"], "VGG16")
    print_top5(resnet50, resnet_inputs, resnet_weights.meta["categories"], "ResNet50")


if __name__ == "__main__":
    main()

Task 02: Architecture Experiments (Alex Net, ResNet101, Mobile Net)

In [ ]:
import torch
from torchvision import models
from torchvision.models import AlexNet_Weights, ResNet101_Weights, MobileNet_V2_Weights
from PIL import Image


def load_image(image_path: str, transform):
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0)
    return tensor


def top5_predictions(model, inputs, labels):
    model.eval()
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.nn.functional.softmax(logits[0], dim=0)
        top_probs, top_idxs = torch.topk(probs, k=5)

    return [(labels[idx], prob.item()) for idx, prob in zip(top_idxs, top_probs)]


def main():
    image_path = input("Enter image path: ").strip()

    experiments = [
        ("AlexNet", AlexNet_Weights.DEFAULT, models.alexnet),
        ("ResNet101", ResNet101_Weights.DEFAULT, models.resnet101),
        ("MobileNetV2", MobileNet_V2_Weights.DEFAULT, models.mobilenet_v2),
    ]

    all_results = {}
    for name, weights, model_builder in experiments:
        model = model_builder(weights=weights)
        inputs = load_image(image_path, weights.transforms())
        labels = weights.meta["categories"]

        preds = top5_predictions(model, inputs, labels)
        all_results[name] = preds

        print(f"\nTop 5 predictions for {name}:")
        for i, (label, prob) in enumerate(preds, start=1):
            print(f"{i}. {label} ({prob * 100:.2f}% probability)")

    print("\nComparison summary:")
    top1 = {name: preds[0] for name, preds in all_results.items()}
    for model_name, (label, prob) in top1.items():
        print(f"- {model_name} top-1: {label} ({prob * 100:.2f}%)")

    unique_top1 = len(set(label for label, _ in top1.values()))
    if unique_top1 == 1:
        print("- All three architectures agree on the same top-1 class.")
    else:
        print("- The architectures disagree on top-1 class, showing architecture-dependent behavior.")


if __name__ == "__main__":
    main()

Task 03: Use your own dataset

In [ ]:
import copy
import os

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_correct += torch.sum(preds == labels).item()
            total += labels.size(0)

    return running_loss / total, running_correct / total


def train(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    best_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_correct += torch.sum(preds == labels).item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = running_correct / total

        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(
            f"Epoch [{epoch}/{epochs}] "
            f"Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} "
            f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    torch.save(model.state_dict(), "best_model.pth")
    print(f"Best validation accuracy: {best_acc:.4f}")
    print("Saved best model to best_model.pth")


def main():
    data_dir = "/source_images"
    batch_size = 32
    epochs = 10
    val_split = 0.2

    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"Dataset directory not found: {data_dir}")

    weights = ResNet18_Weights.DEFAULT
    transform = weights.transforms()

    dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    if len(dataset) < 2:
        raise ValueError("Dataset must contain at least 2 images to perform train/val split.")

    val_size = max(1, int(len(dataset) * val_split))
    train_size = len(dataset) - val_size
    train_set, val_set = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = models.resnet18(weights=weights)
    for param in model.parameters():
        param.requires_grad = False

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, len(dataset.classes))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

    train(model, train_loader, val_loader, criterion, optimizer, device, epochs)


if __name__ == "__main__":
    main()